# Sample comparison (report-ready)

This notebook builds report-ready results across samples to support the thesis aims.

**Scope**
- Pipeline validation and rate summaries (Aim 1)
- Deletion vs insertion patterns, size distributions, and tandem duplications (Aim 2)
- Paternal age effect trends (Aim 3)

Retrotransposition-specific analyses are intentionally excluded for now.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

# === Configuration (edit here) ===
RESULTS_ROOT = Path("/home/peterkad/pkadmaster/indel_scanner/results")
SAMPLES = [
    "tr_plus_unmapped_diploid_v2",
    "ph_plus_unmapped_diploid_v2",
	"chk_plus_unmapped_diploid_v2",
	"da1_plus_unmapped_diploid_v2",
	"la_plus_unmapped_diploid_v2",
	"tsaed_plus_unmapped_diploid_v2"
]

SAMPLE_METADATA = pd.DataFrame(
    [
        {
            "sample": "tr_plus_unmapped_diploid_v2",
            "donor_id": "tr",
            "paternal_age": 35,
        },
        {
            "sample": "ph_plus_unmapped_diploid_v2",
            "donor_id": "ph",
            "paternal_age": 35,
        },
        {
            "sample": "chk_plus_unmapped_diploid_v2",
            "donor_id": "chk",
            "paternal_age": 25,
        },
        {
            "sample": "da1_plus_unmapped_diploid_v2",
            "donor_id": "da1",
            "paternal_age": 49,
        },
        {
            "sample": "la_plus_unmapped_diploid_v2",
            "donor_id": "la",
            "paternal_age": 25,
        },
        {
            "sample": "tsaed_plus_unmapped_diploid_v2",
            "donor_id": "tsaed",
            "paternal_age": 35,
        },
    ]
)
DONOR_MAP = SAMPLE_METADATA.set_index("sample")["donor_id"].to_dict()


def with_sample_label(df: pd.DataFrame) -> pd.DataFrame:
    if "sample" not in df.columns:
        return df
    return df.assign(sample_label=df["sample"].map(DONOR_MAP).fillna(df["sample"]))


FIGURES_DIR = Path("figures")
TABLES_DIR = Path("tables")
FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)

SHORT_INS_MAX_BP = 10
SIZE_BIN_ORDER = ["1bp", "2-3bp", "4-10bp", ">10bp"]

In [3]:
def latest_run_dir(sample_root: Path, required_file: str | None = None) -> Path | None:
    if not sample_root.exists():
        return None
    run_dirs = [p for p in sample_root.iterdir() if p.is_dir()]
    if not run_dirs:
        return None
    if required_file:
        with_file = [p for p in run_dirs if (p / required_file).exists()]
        if with_file:
            return max(with_file, key=lambda p: p.stat().st_mtime)
    return max(run_dirs, key=lambda p: p.stat().st_mtime)


def read_tsv(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    return pd.read_csv(path, sep="\t")


def read_json(path: Path) -> dict | None:
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def save_table(df: pd.DataFrame, name: str) -> None:
    out_path = TABLES_DIR / f"{name}.csv"
    df.to_csv(out_path, index=False)


def save_figure(fig: plt.Figure, name: str) -> None:
    fig.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches="tight")


def parse_sequence_context(context: str) -> tuple[str, str, str] | None:
    match = re.match(r"^(.*)\[(.*)\](.*)$", str(context))
    if not match:
        return None
    return match.group(1), match.group(2), match.group(3)


def is_tandem_dup(prefix: str, ins: str, suffix: str) -> bool:
    if not ins:
        return False
    prefix = prefix.upper()
    suffix = suffix.upper()
    ins = ins.upper()
    return prefix.endswith(ins) or suffix.startswith(ins)


def assign_size_bin(length: int) -> str:
    if length == 1:
        return "1bp"
    if 2 <= length <= 3:
        return "2-3bp"
    if 4 <= length <= 10:
        return "4-10bp"
    return ">10bp"


records = []
missing = []

for sample in SAMPLES:
    sample_root = RESULTS_ROOT / sample

    pipeline_run = latest_run_dir(
        sample_root, required_file="per_type_mutation_frequency.tsv"
    )
    callable_run = latest_run_dir(sample_root, required_file="callable_stats_1_10.json")

    if pipeline_run is None:
        missing.append((sample, "no_pipeline_run"))
        continue

    per_type = pipeline_run / "per_type_mutation_frequency.tsv"
    callable_bases = pipeline_run / "callable_bases.tsv"
    passed_indels = pipeline_run / "processed" / "final_passed_indels.tsv"

    callable_stats = None
    if callable_run is not None:
        callable_stats = callable_run / "callable_stats_1_10.json"

    if not per_type.exists() or not callable_bases.exists():
        missing.append((sample, str(pipeline_run)))
        continue

    records.append(
        {
            "sample": sample,
            "pipeline_run": pipeline_run,
            "callable_run": callable_run,
            "per_type": per_type,
            "callable_bases": callable_bases,
            "passed_indels": passed_indels if passed_indels.exists() else None,
            "callable_stats": callable_stats if callable_stats and callable_stats.exists() else None,
        }
    )

pd.DataFrame(records), pd.DataFrame(missing, columns=["sample", "issue"])

(Empty DataFrame
 Columns: []
 Index: [],
                            sample       issue
 0     tr_plus_unmapped_diploid_v2  no_run_dir
 1     ph_plus_unmapped_diploid_v2  no_run_dir
 2    chk_plus_unmapped_diploid_v2  no_run_dir
 3    da1_plus_unmapped_diploid_v2  no_run_dir
 4     la_plus_unmapped_diploid_v2  no_run_dir
 5  tsaed_plus_unmapped_diploid_v2  no_run_dir)

In [ ]:
def load_per_type(path: Path, sample: str) -> pd.DataFrame:
    df = read_tsv(path)
    if df is None:
        return pd.DataFrame()
    df["sample"] = sample
    return df


def load_callable(path: Path, sample: str) -> pd.DataFrame:
    df = read_tsv(path)
    if df is None:
        return pd.DataFrame()
    df["sample"] = sample
    return df


def load_passed_indels(path: Path | None, sample: str) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()
    df = read_tsv(path)
    if df is None:
        return pd.DataFrame()
    df["sample"] = sample
    return df


per_type_dfs = []
callable_dfs = []
passed_dfs = []

for rec in records:
    sample = rec["sample"]
    per_type_dfs.append(load_per_type(rec["per_type"], sample))
    callable_dfs.append(load_callable(rec["callable_bases"], sample))
    passed_dfs.append(load_passed_indels(rec["passed_indels"], sample))

per_type_all = pd.concat(per_type_dfs, ignore_index=True) if per_type_dfs else pd.DataFrame()
callable_all = pd.concat(callable_dfs, ignore_index=True) if callable_dfs else pd.DataFrame()
passed_all = pd.concat(passed_dfs, ignore_index=True) if passed_dfs else pd.DataFrame()

per_type_all.head()

In [ ]:
## Aim 1 — Pipeline validation and basic rate summaries

Goal: confirm the pipeline outputs are consistent across samples and produce stable callable bases and rate estimates.

In [ ]:
if per_type_all.empty or callable_all.empty:
    print("No per-type or callable data found. Check RESULTS_ROOT/SAMPLES.")
else:
    callable_subset = callable_all.copy()
    callable_subset["size_bp"] = callable_subset["mutation_type"].str.extract(
        r"len_(\d+)bp"
    )[0]
    callable_subset = callable_subset.dropna(subset=["size_bp"]).copy()
    callable_subset["size_bp"] = callable_subset["size_bp"].astype(int)
    callable_subset = callable_subset[callable_subset["size_bp"] <= 10].copy()

    callable_summary = (
        callable_subset.groupby(["sample", "mutation_type"], as_index=False)[
            "callable_bases"
        ]
        .sum()
        .rename(columns={"callable_bases": "callable_bases_total"})
    )

    stats_rows = []
    for rec in records:
        stats = read_json(rec["callable_stats"]) if rec.get("callable_stats") else None
        if not stats:
            continue
        totals = stats.get("sampling_totals_by_type", {})
        passable = stats.get("sampling_passable_by_type", {})
        for mutation_type, total in totals.items():
            if total == 0:
                continue
            pass_count = passable.get(mutation_type, 0)
            stats_rows.append(
                {
                    "sample": rec["sample"],
                    "mutation_type": mutation_type,
                    "sampling_total": total,
                    "sampling_passable": pass_count,
                    "class_pct": (pass_count / total) * 100,
                }
            )
    stats_pct = pd.DataFrame(stats_rows)

    callable_summary["callable_bases_mbp"] = (
        callable_summary["callable_bases_total"] / 1_000_000
    )
    if not stats_pct.empty:
        callable_summary = callable_summary.merge(
            stats_pct, on=["sample", "mutation_type"], how="left"
        )
    else:
        callable_summary["class_pct"] = np.nan
        callable_summary["sampling_total"] = np.nan
        callable_summary["sampling_passable"] = np.nan

    callable_summary["indel_type"] = callable_summary["mutation_type"].str.extract(
        r"^(ins|del)"
    )
    callable_summary["size_bp"] = callable_summary["mutation_type"].str.extract(
        r"len_(\d+)bp"
    )[0].astype(int)

    callable_summary = callable_summary[
        [
            "sample",
            "mutation_type",
            "indel_type",
            "size_bp",
            "callable_bases_mbp",
            "sampling_total",
            "sampling_passable",
            "class_pct",
        ]
    ]

    callable_summary_mbp = callable_summary.pivot(
        index="mutation_type",
        columns="sample",
        values="callable_bases_mbp",
    )
    callable_summary_pct = callable_summary.pivot(
        index="mutation_type",
        columns="sample",
        values="class_pct",
    )

    save_table(
        callable_summary_mbp.round(2).reset_index(),
        "aim1_callable_class_mbp_non_str_1_10",
    )
    if not stats_pct.empty:
        save_table(
            callable_summary_pct.round(3).reset_index(),
            "aim1_callable_class_pct_non_str_1_10",
        )
    else:
        print("Missing callable_stats_1_10.json; % callable table will be empty.")

    verify_cols = [
        "sample",
        "mutation_type",
        "sampling_passable",
        "sampling_total",
        "class_pct",
    ]
    verify_table = callable_summary[verify_cols].copy()
    save_table(
        verify_table.round({"class_pct": 3}),
        "aim1_callable_class_pct_verify_non_str_1_10",
    )
    display(verify_table)

    def format_heatmap(df: pd.DataFrame, indel_type: str) -> pd.DataFrame:
        subset = callable_summary[callable_summary["indel_type"] == indel_type].copy()
        subset = subset.dropna(subset=["class_pct"]).copy()
        subset = with_sample_label(subset)
        subset = subset.sort_values(["size_bp"], ascending=False)
        pivot = subset.pivot(
            index="size_bp",
            columns="sample_label",
            values="class_pct",
        )
        return pivot

    heatmap_ins = format_heatmap(callable_summary, "ins")
    heatmap_del = format_heatmap(callable_summary, "del")

    if heatmap_ins.empty or heatmap_del.empty:
        print(
            "Missing callable_stats_1_10.json for one or more samples; "
            "skipping callable-class heatmap."
        )
    else:
        combined_min = np.nanmin(
            pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        )
        combined_max = np.nanmax(
            pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        )

        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(6.8, 6.6),
            sharex=True,
            gridspec_kw={"hspace": 0.06},
        )
        fig.subplots_adjust(right=0.86)
        cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

        sns.heatmap(
            heatmap_del,
            cmap="viridis",
            linewidths=0.2,
            linecolor="white",
            vmin=combined_min,
            vmax=combined_max,
            cbar=False,
            ax=axes[0],
        )
        axes[0].set_title("")
        axes[0].set_ylabel("Deletions (bp)")
        axes[0].set_xlabel("")

        sns.heatmap(
            heatmap_ins,
            cmap="viridis",
            linewidths=0.2,
            linecolor="white",
            vmin=combined_min,
            vmax=combined_max,
            cbar=True,
            cbar_ax=cbar_ax,
            cbar_kws={"label": "% callable"},
            ax=axes[1],
        )
        axes[1].set_title("")
        axes[1].set_ylabel("Insertions (bp)")
        axes[1].set_xlabel("Sample")

        fig.suptitle("Non-STR 1–10bp callable fraction (% callable)")
        fig.tight_layout(rect=[0, 0, 0.86, 0.96])
        save_figure(fig, "aim1_callable_class_pct_heatmap_non_str_1_10")
        plt.close(fig)

    indel_summary = per_type_all.groupby("sample", as_index=False).agg(
        total_indels=("count", "sum")
    )
    if "unique_sites" in per_type_all.columns:
        unique_summary = (
            per_type_all.groupby("sample", as_index=False)["unique_sites"].sum()
        )
        indel_summary = indel_summary.merge(unique_summary, on="sample", how="left")
        indel_summary = indel_summary.rename(
            columns={"unique_sites": "total_unique_sites"}
        )
    save_table(indel_summary, "aim1_indel_summary")
    display(indel_summary)

    str_counts = (
        per_type_all.groupby(["sample", "str_class"], as_index=False)["count"].sum()
    )
    save_table(str_counts, "aim1_str_vs_non_str_counts")

    plot_callable = callable_all.copy()
    plot_callable["region_class"] = np.where(
        plot_callable["mutation_type"].str.contains("motif"),
        "STR",
        "non_STR",
    )
    plot_callable = (
        plot_callable.groupby(["sample", "region_class"], as_index=False)[
            "callable_bases"
        ].sum()
    )
    plot_callable["callable_bases_mbp"] = plot_callable["callable_bases"] / 1_000_000
    plot_callable = with_sample_label(plot_callable)

    fig, ax = plt.subplots(figsize=(6.8, 3.6))
    sns.barplot(
        data=plot_callable,
        x="sample_label",
        y="callable_bases_mbp",
        hue="region_class",
        ax=ax,
    )
    ax.set_ylabel("Callable bases (Mbp)")
    ax.set_xlabel("Sample")
    ax.legend(title="Class")
    fig.suptitle("Callable bases by region class")
    fig.tight_layout()
    save_figure(fig, "aim1_callable_bases")
    plt.close(fig)

    str_counts = with_sample_label(str_counts)

    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    sns.barplot(
        data=str_counts,
        x="sample_label",
        y="count",
        hue="str_class",
        ax=ax,
    )
    ax.set_ylabel("Indel count")
    ax.set_xlabel("Sample")
    ax.legend(title="Region class")
    fig.suptitle("STR vs non-STR indel counts")
    fig.tight_layout()
    save_figure(fig, "aim1_str_vs_non_str_counts")
    plt.close(fig)

    callable_summary

In [ ]:
# Non-STR 1–10bp per-class mutation rates (heatmap)
non_str_rates = per_type_all.copy()
non_str_rates = non_str_rates[non_str_rates["str_class"] == "non_STR"].copy()
non_str_rates["size_bp"] = non_str_rates["mutation_type"].str.extract(r"len_(\d+)bp")[0]
non_str_rates = non_str_rates.dropna(subset=["size_bp"]).copy()
non_str_rates["size_bp"] = non_str_rates["size_bp"].astype(int)
non_str_rates = non_str_rates[non_str_rates["size_bp"] <= 10].copy()
non_str_rates["indel_type"] = non_str_rates["mutation_type"].str.extract(r"^(ins|del)")

non_str_rates = with_sample_label(non_str_rates)

rate_ins = non_str_rates[non_str_rates["indel_type"] == "ins"].copy()
rate_del = non_str_rates[non_str_rates["indel_type"] == "del"].copy()

rate_ins = rate_ins.sort_values(["size_bp"], ascending=False)
rate_del = rate_del.sort_values(["size_bp"], ascending=False)

heatmap_ins = rate_ins.pivot(
    index="size_bp",
    columns="sample_label",
    values="frequency",
)
heatmap_del = rate_del.pivot(
    index="size_bp",
    columns="sample_label",
    values="frequency",
)

heatmap_ins = heatmap_ins.where(heatmap_ins > 0)
heatmap_del = heatmap_del.where(heatmap_del > 0)

combined_min = np.nanmin(
    pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
)
combined_max = np.nanmax(
    pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
)

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(6.8, 6.6),
    sharex=True,
    gridspec_kw={"hspace": 0.06},
)
fig.subplots_adjust(right=0.86)
cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

sns.heatmap(
    heatmap_del,
    cmap="viridis",
    linewidths=0.2,
    linecolor="white",
    norm=LogNorm(vmin=combined_min, vmax=combined_max),
    cbar=False,
    ax=axes[0],
)
axes[0].set_title("")
axes[0].set_ylabel("Deletions (bp)")
axes[0].set_xlabel("")

sns.heatmap(
    heatmap_ins,
    cmap="viridis",
    linewidths=0.2,
    linecolor="white",
    norm=LogNorm(vmin=combined_min, vmax=combined_max),
    cbar=True,
    cbar_ax=cbar_ax,
    cbar_kws={"label": "Mutation rate (per base, log scale)"},
    ax=axes[1],
)
axes[1].set_title("")
axes[1].set_ylabel("Insertions (bp)")
axes[1].set_xlabel("Sample")

fig.suptitle("Non-STR 1–10bp per-class mutation rates")
fig.tight_layout(rect=[0, 0, 0.86, 0.96])
save_figure(fig, "aim1_non_str_per_class_rates_heatmap")
plt.close(fig)


In [ ]:
**Interpretation (Aim 1):**
The non‑STR 1–10bp callable‑class heatmap uses per‑class sampling denominators to show comparable interrogation rates across samples and highlight sample‑specific gaps or outliers.

**Comparability notes:**
- Heatmap values use `% callable` from `sampling_passable_by_type / sampling_totals_by_type`.
- Per‑class rates in `per_type_mutation_frequency.tsv` are normalized by class‑specific callable bases, so those rates are comparable across samples within the same class.
- Totals across classes are not directly comparable because callable bases are not additive across classes.

In [ ]:
## Aim 2 — Indel mechanisms (no retrotransposition)

Goal: compare insertion vs deletion rates, size distributions, and tandem duplication signature in short insertions.

In [ ]:
if per_type_all.empty:
    print("No per-type data available for Aim 2.")
else:
    per_type_all = per_type_all.copy()
    per_type_all["indel_type"] = per_type_all["mutation_type"].str.extract(r"^(ins|del)")

    non_str_rates = per_type_all[per_type_all["str_class"] == "non_STR"].copy()
    non_str_rates["size_bp"] = non_str_rates["mutation_type"].str.extract(
        r"len_(\d+)bp"
    )[0]
    non_str_rates = non_str_rates.dropna(subset=["size_bp"]).copy()
    non_str_rates["size_bp"] = non_str_rates["size_bp"].astype(int)
    non_str_rates = non_str_rates[non_str_rates["size_bp"] <= 10].copy()
    non_str_rates["indel_type"] = non_str_rates["mutation_type"].str.extract(
        r"^(ins|del)"
    )
    non_str_rates = with_sample_label(non_str_rates)

    rate_ins = non_str_rates[non_str_rates["indel_type"] == "ins"].copy()
    rate_del = non_str_rates[non_str_rates["indel_type"] == "del"].copy()
    rate_ins = rate_ins.sort_values(["size_bp"], ascending=False)
    rate_del = rate_del.sort_values(["size_bp"], ascending=False)

    heatmap_ins = rate_ins.pivot(
        index="size_bp",
        columns="sample_label",
        values="frequency",
    )
    heatmap_del = rate_del.pivot(
        index="size_bp",
        columns="sample_label",
        values="frequency",
    )

    heatmap_ins = heatmap_ins.where(heatmap_ins > 0)
    heatmap_del = heatmap_del.where(heatmap_del > 0)

    combined_min = np.nanmin(
        pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
    )
    combined_max = np.nanmax(
        pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
    )

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(6.8, 6.6),
        sharex=True,
        gridspec_kw={"hspace": 0.06},
    )
    fig.subplots_adjust(right=0.86)
    cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

    sns.heatmap(
        heatmap_del,
        cmap="viridis",
        linewidths=0.2,
        linecolor="white",
        norm=LogNorm(vmin=combined_min, vmax=combined_max),
        cbar=False,
        ax=axes[0],
    )
    axes[0].set_title("")
    axes[0].set_ylabel("Deletions (bp)")
    axes[0].set_xlabel("")

    sns.heatmap(
        heatmap_ins,
        cmap="viridis",
        linewidths=0.2,
        linecolor="white",
        norm=LogNorm(vmin=combined_min, vmax=combined_max),
        cbar=True,
        cbar_ax=cbar_ax,
        cbar_kws={"label": "Mutation rate (per base, log scale)"},
        ax=axes[1],
    )
    axes[1].set_title("")
    axes[1].set_ylabel("Insertions (bp)")
    axes[1].set_xlabel("Sample")

    fig.suptitle("Non-STR 1–10bp mutation rates (per class)")
    fig.tight_layout(rect=[0, 0, 0.86, 0.96])
    save_figure(fig, "aim2_non_str_rate_heatmap_1_10")
    plt.close(fig)

    if not passed_all.empty and "length" in passed_all.columns and "type" in passed_all.columns:
        passed_all = passed_all.copy()
        passed_all = passed_all[passed_all["type"].isin(["ins", "del"])].copy()
        passed_all["length"] = pd.to_numeric(passed_all["length"], errors="coerce")
        passed_all = passed_all.dropna(subset=["length"]) 
        passed_all["length"] = passed_all["length"].astype(int)
        passed_all["size_bin"] = passed_all["length"].apply(assign_size_bin)

        if {"in_STR_region", "in_STR"}.issubset(passed_all.columns):
            non_str_mask = ~(passed_all["in_STR_region"] & passed_all["in_STR"])
        elif "in_STR" in passed_all.columns:
            non_str_mask = ~passed_all["in_STR"]
        else:
            non_str_mask = pd.Series([True] * len(passed_all), index=passed_all.index)

        passed_non_str = passed_all[non_str_mask].copy()

        size_counts = (
            passed_non_str.groupby(["sample", "type", "size_bin"], as_index=False)
            .size()
            .rename(columns={"size": "count"})
        )
        size_counts["fraction"] = size_counts["count"] / size_counts.groupby(
            ["sample", "type"]
        )["count"].transform("sum")
        size_counts["size_bin"] = pd.Categorical(
            size_counts["size_bin"], categories=SIZE_BIN_ORDER, ordered=True
        )
        size_counts = size_counts.sort_values(["type", "size_bin"])
        save_table(size_counts, "aim2_size_distribution_non_str")

        size_counts = with_sample_label(size_counts)
        ins_size = size_counts[size_counts["type"] == "ins"].copy()
        del_size = size_counts[size_counts["type"] == "del"].copy()

        heatmap_ins = ins_size.pivot(
            index="size_bin",
            columns="sample_label",
            values="fraction",
        )
        heatmap_del = del_size.pivot(
            index="size_bin",
            columns="sample_label",
            values="fraction",
        )

        combined_min = np.nanmin(
            pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        )
        combined_max = np.nanmax(
            pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        )

        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(6.8, 6.6),
            sharex=True,
            gridspec_kw={"hspace": 0.06},
        )
        fig.subplots_adjust(right=0.86)
        cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

        sns.heatmap(
            heatmap_del,
            cmap="viridis",
            linewidths=0.2,
            linecolor="white",
            vmin=combined_min,
            vmax=combined_max,
            cbar=False,
            ax=axes[0],
        )
        axes[0].set_title("")
        axes[0].set_ylabel("Deletions")
        axes[0].set_xlabel("")

        sns.heatmap(
            heatmap_ins,
            cmap="viridis",
            linewidths=0.2,
            linecolor="white",
            vmin=combined_min,
            vmax=combined_max,
            cbar=True,
            cbar_ax=cbar_ax,
            cbar_kws={"label": "Fraction within type"},
            ax=axes[1],
        )
        axes[1].set_title("")
        axes[1].set_ylabel("Insertions")
        axes[1].set_xlabel("Sample")

        fig.suptitle("Non-STR size distribution (fraction within type)")
        fig.tight_layout(rect=[0, 0, 0.86, 0.96])
        save_figure(fig, "aim2_size_distribution_non_str")
        plt.close(fig)

        context_col = "[sequence]_context"
        if context_col in passed_non_str.columns:
            ins_df = passed_non_str[
                (passed_non_str["type"] == "ins")
                & (passed_non_str["length"] <= SHORT_INS_MAX_BP)
            ].copy()
            if not ins_df.empty:
                parsed = ins_df[context_col].apply(parse_sequence_context)
                ins_df["prefix"] = parsed.apply(lambda x: x[0] if x else "")
                ins_df["ins_seq"] = parsed.apply(lambda x: x[1] if x else "")
                ins_df["suffix"] = parsed.apply(lambda x: x[2] if x else "")
                ins_df["is_tandem_dup"] = ins_df.apply(
                    lambda r: is_tandem_dup(r["prefix"], r["ins_seq"], r["suffix"]),
                    axis=1,
                )

                tandem_summary = (
                    ins_df.groupby("sample", as_index=False)
                    .agg(
                        short_insertions=("is_tandem_dup", "size"),
                        tandem_dups=("is_tandem_dup", "sum"),
                    )
                )
                tandem_summary["fraction_tandem"] = (
                    tandem_summary["tandem_dups"] / tandem_summary["short_insertions"]
                )
                save_table(tandem_summary, "aim2_tandem_duplications")

                tandem_summary = with_sample_label(tandem_summary)

                fig, ax = plt.subplots(figsize=(6.2, 3.4))
                sns.barplot(
                    data=tandem_summary,
                    x="sample_label",
                    y="fraction_tandem",
                    ax=ax,
                )
                ax.set_ylabel("Fraction tandem duplication")
                ax.set_xlabel("Sample")
                fig.suptitle("Short insertions consistent with tandem duplication")
                fig.tight_layout()
                save_figure(fig, "aim2_tandem_dup_fraction")
                plt.close(fig)

        non_str_rates.head()

In [ ]:
**Interpretation (Aim 2):**
Non‑STR rate contrasts, size distributions, and tandem‑duplication fractions directly test whether insertion and deletion processes are mechanistically distinct.

**Comparability notes:**
- The per‑class rate heatmap uses `per_type_mutation_frequency.tsv`, which is normalized by class‑specific callable bases, so rates are comparable across samples within the same class.
- Size‑distribution heatmaps use fractions within each sample and type, so they compare shape rather than absolute rates.
- Tandem‑duplication fractions are computed within short insertions per sample, making them comparable across samples as proportions.

## Aim 3 — Paternal age effect

Goal: compare mutation rates and insertion:deletion ratios across donors and relate to paternal age.

In [ ]:
if per_type_all.empty:
    print("No per-type data available for Aim 3.")
else:
    per_type_all = per_type_all.copy()
    per_type_all["indel_type"] = per_type_all["mutation_type"].str.extract(r"^(ins|del)")

    per_sample_counts = (
        per_type_all.groupby(["sample", "indel_type"], as_index=False)["count"].sum()
    )
    per_sample_callable = (
        per_type_all.groupby("sample", as_index=False)["callable_bases"].sum()
    )

    per_sample_rates = per_sample_counts.pivot_table(
        index="sample", columns="indel_type", values="count", aggfunc="sum"
    ).reset_index()
    per_sample_rates = per_sample_rates.merge(per_sample_callable, on="sample", how="left")
    per_sample_rates = per_sample_rates.rename(columns={"callable_bases": "callable_bases_total"})

    per_sample_rates["total_indels"] = per_sample_rates[["ins", "del"]].sum(axis=1)
    per_sample_rates["mutation_rate"] = (
        per_sample_rates["total_indels"] / per_sample_rates["callable_bases_total"]
    )
    per_sample_rates["ins_del_ratio"] = per_sample_rates["ins"] / per_sample_rates["del"]

    aim3_table = per_sample_rates.merge(SAMPLE_METADATA, on="sample", how="left")
    aim3_table = aim3_table[
        [
            "sample",
            "donor_id",
            "paternal_age",
            "callable_bases_total",
            "mutation_rate",
            "ins_del_ratio",
        ]
    ]
    save_table(aim3_table, "aim3_paternal_age_summary")
    display(aim3_table)

    age_plot_df = aim3_table.dropna(subset=["paternal_age", "mutation_rate"])
    if len(age_plot_df) >= 2:
        fig, ax = plt.subplots(figsize=(5.8, 3.4))
        sns.regplot(
            data=age_plot_df,
            x="paternal_age",
            y="mutation_rate",
            ax=ax,
            scatter_kws={"s": 50},
            line_kws={"color": "black"},
        )
        ax.set_xlabel("Paternal age")
        ax.set_ylabel("Mutation rate")
        fig.suptitle("Mutation rate vs paternal age")
        fig.tight_layout()
        save_figure(fig, "aim3_rate_vs_age")
        plt.close(fig)
    else:
        print("Add paternal ages in SAMPLE_METADATA to enable the Aim 3 plot.")

    aim3_table

**Interpretation (Aim 3):**
The rate-versus-age comparison and insertion:deletion ratios test whether the donors show the expected paternal age effect trend.